# Bi-Hemispheric Neuromorphic AI: Llama 3.1 8B + Sapient HRM + ARC-AGI-1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sneed-and-feed/brain-ai/blob/main/notebooks/02_bihemispheric_llama_arc_colab.ipynb)

This notebook connects the **complete multi-modal bi-hemispheric architecture** on **Google Colab Pro+ (A100 80GB)**:
1. **Left Hemisphere ($\mathcal{H}_L$)**: `meta-llama/Llama-3.1-8B-Instruct` (BF16, residual stream hooked at Layer 16).
2. **Right Hemisphere ($\mathcal{H}_R$)**: `Sapient HRM 27M` (dual-timescale recurrent spatial engine).
3. **Corpus Callosum ($\mathcal{C}_{LR}$)**: Dale-constrained Differential Cross-Attention bridge with Rajan-Abbott spectral stability.
4. **Computational Amygdala ($\mathcal{A}$)**: Fast sub-25ms affective router and neuromodulatory gain controller.
5. **Benchmark**: `ARC-AGI-1` inductive spatial abstraction challenges.

**Objective**: Prove dual-brain cognitive synergy where the Left Hemisphere reasons symbolically about rules and the Right Hemisphere unrolls spatial latent constraints to solve ARC grids without token-by-token backtracking.

## 1. Environment & GPU Verification (Colab Pro+ A100)

In [ ]:
# Check GPU hardware (NVIDIA A100 80GB recommended)
!nvidia-smi

# Idempotent clone or enter repository
import os
if not os.path.exists("brain-ai") and not os.getcwd().endswith("brain-ai"):
    !git clone https://github.com/sneed-and-feed/brain-ai.git
    %cd brain-ai
elif os.path.exists("brain-ai"):
    %cd brain-ai

!pip install -q torch transformers accelerate peft einops matplotlib scipy huggingface_hub

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

# Authenticate with Hugging Face using Colab secrets (HF_TOKEN required for Llama 3.1 8B)
try:
    from google.colab import userdata
    from huggingface_hub import login
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        login(token=hf_token)
        print("Successfully authenticated with Hugging Face via Colab secrets (HF_TOKEN)!")
    else:
        print("Warning: HF_TOKEN secret not found. Add HF_TOKEN under Colab Secrets to access Llama 3.1 8B.")
except Exception as e:
    print("Note on HF authentication:", e)

### Sync Latest Code from GitHub (`git pull`)

In [ ]:
import os, sys
if os.path.exists("/content/brain-ai") and not os.getcwd().endswith("brain-ai"):
    %cd /content/brain-ai
elif os.path.exists("brain-ai") and not os.getcwd().endswith("brain-ai"):
    %cd brain-ai

!git pull origin main

for mod in list(sys.modules.keys()):
    if mod.startswith("brain_ai"):
        del sys.modules[mod]

print("Synced with origin/main and reloaded brain_ai modules!")

## 2. Ingest ARC-AGI-1 Tasks

Loads official ARC-AGI-1 tasks with automatic caching. Each task contains 2–5 training demonstration pairs and a test challenge grid.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
from brain_ai.tasks.arc import ARCDataset

# Official ARC-AGI 10-color discrete palette
ARC_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25",
    "#FFFFFF" # 10: Padding
]
arc_cmap = ListedColormap(ARC_COLORS)

arc_dataset = ARCDataset(cache_dir="data/arc", max_grid_size=15)
sample_batch = arc_dataset.get_batch(batch_size=1)

H, W = sample_batch.target_shapes[0]
test_in_grid = sample_batch.test_inputs[0, :H, :W].cpu().numpy()
test_out_grid = sample_batch.test_targets[0, :H, :W].cpu().numpy()

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(test_in_grid, cmap=arc_cmap, vmin=0, vmax=10)
ax[0].set_title(f"ARC Test Input Grid ({H}x{W})")
ax[0].axis("off")

ax[1].imshow(test_out_grid, cmap=arc_cmap, vmin=0, vmax=10)
ax[1].set_title(f"ARC Ground Truth Output Grid ({H}x{W})")
ax[1].axis("off")

plt.tight_layout()
plt.show()

print("Sample Left Hemisphere Prompt:")
print(sample_batch.text_prompts[0])

## 3. Instantiate Bi-Hemispheric System: Llama 3.1 8B + HRM + Callosum

We load:
- `LeftHemisphereLlama`: Llama 3.1 8B in BF16 hooked at Layer 16 ($d_{\text{LH}} = 4096$).
- `HierarchicalReasoningModel`: Sapient HRM 27M ($d_{\text{RH}} = 512$).
- `CorpusCallosum`: Dale-constrained Differential Cross-Attention ($d_{\text{call}} = 512, s = -1.0$).
- `OpenJevAmygdalarRouter`: Affective salience & gating controller.

In [ ]:
import torch.nn as nn
from brain_ai.models.llama_lh import LeftHemisphereLlama
from brain_ai.models.ensemble import BiHemisphericBrain

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target Execution Device: {device.upper()}")

# 1. Initialize Left Hemisphere (Llama 3.1 8B Instruct)
try:
    lh_model = LeftHemisphereLlama(
        model_id="meta-llama/Llama-3.1-8B-Instruct",
        hook_layer=16,
        d_model=4096,
        torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
        device=device,
        mock_mode=False
    )
except Exception as e:
    print("Could not load live Llama 3.1 8B (check HF_TOKEN in secrets). Falling back to mock mode:", e)
    lh_model = LeftHemisphereLlama(d_model=4096, mock_mode=True, device=device)

# 2. Initialize Bi-Hemispheric Brain Ensemble
brain = BiHemisphericBrain(
    d_lh=4096,          # Llama 3.1 8B residual dimension
    d_rh=512,           # HRM 27M latent dimension
    d_callosum=512,     # Callosal manifold dimension
    callosal_heads=4,
    hrm_cycles=3,
    hrm_max_segments=6
).to(device)

print("Bi-Hemispheric System Successfully Instantiated!")

## 4. Multi-Modal ARC-AGI Spatial Head & Embeddings

The Right Hemisphere processes ARC grid tokens (10 discrete colors + coordinate embeddings) and predicts 10-class color distributions for each cell in the target grid.

In [ ]:
# ARC Spatial Embedder and Multi-Class Grid Prediction Head
class ARCSpatialGridEmbedding(nn.Module):
    def __init__(self, num_colors=11, d_model=512, max_size=32):
        super().__init__()
        self.color_embed = nn.Embedding(num_colors, d_model)
        self.row_embed = nn.Embedding(max_size, d_model)
        self.col_embed = nn.Embedding(max_size, d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, grids: torch.Tensor) -> torch.Tensor:
        # grids: [B, H, W]
        B, H, W = grids.shape
        device = grids.device
        rows = torch.arange(H, device=device).unsqueeze(1).repeat(1, W).view(-1)
        cols = torch.arange(W, device=device).unsqueeze(0).repeat(H, 1).view(-1)
        
        flat_tokens = grids.view(B, H * W)
        color_emb = self.color_embed(flat_tokens)
        pos_emb = self.row_embed(rows) + self.col_embed(cols)
        return self.proj(color_emb + pos_emb.unsqueeze(0))

class ARCPredictionHead(nn.Module):
    def __init__(self, d_model=512, num_colors=10, max_size=15):
        super().__init__()
        self.max_size = max_size
        self.conv = nn.Sequential(
            nn.Conv2d(d_model, 128, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, num_colors, kernel_size=1)
        )

    def forward(self, rh_latents: torch.Tensor) -> torch.Tensor:
        # rh_latents: [B, H*W, d_model]
        B, L, D = rh_latents.shape
        H = W = self.max_size
        # Guarantee strict contiguous memory layout for cuDNN convolution
        x_2d = rh_latents.transpose(1, 2).contiguous().view(B, D, H, W)
        logits_2d = self.conv(x_2d) # [B, 10, H, W]
        return logits_2d

arc_embedder = ARCSpatialGridEmbedding(num_colors=11, d_model=512, max_size=32).to(device)
arc_head = ARCPredictionHead(d_model=512, num_colors=10, max_size=15).to(device)
print("ARC Spatial Embedder & Head Ready!")

## 5. End-to-End Bi-Hemispheric Synergy Training

We train the Corpus Callosum bridge, HRM recurrent engine, and ARC heads:
1. Llama extracts intermediate residual representations from the natural language prompt.
2. HRM unrolls spatial latent recurrence on the visual grid.
3. Dale-constrained transcallosal differential attention couples language & spatial representations.

In [ ]:
import torch.optim as optim

trainable_params = (
    list(arc_embedder.parameters()) +
    list(brain.corpus_callosum.parameters()) +
    list(brain.proj_lh_to_call.parameters()) +
    list(brain.proj_rh_to_call.parameters()) +
    list(brain.proj_call_to_lh.parameters()) +
    list(brain.proj_call_to_rh.parameters()) +
    list(brain.right_hemisphere.parameters()) +
    list(arc_head.parameters())
)

optimizer = optim.AdamW(trainable_params, lr=3e-4, weight_decay=1e-4)
ce_loss_fn = nn.CrossEntropyLoss(ignore_index=10, reduction="none")

history = {"task_loss": [], "homeo_loss": [], "exact_match": [], "spec_radius": []}
num_steps = 150
batch_size = 4

print("Starting Multi-Modal Bi-Hemispheric Training on ARC-AGI...")
for step in range(1, num_steps + 1):
    batch = arc_dataset.get_batch(batch_size=batch_size, device=device)
    test_in = batch.test_inputs
    test_target = batch.test_targets
    mask = batch.target_masks
    
    # 1. Left Hemisphere: Tokenize prompt and extract residual stream
    tokenized = lh_model.tokenize(batch.text_prompts, max_length=128)
    z_lh = lh_model.extract_residual_stream(tokenized["input_ids"], tokenized["attention_mask"]).to(torch.float32)
    
    # 2. Right Hemisphere: 2D Spatial Grid Embedding
    rh_in = arc_embedder(test_in)
    
    # 3. Synchronized Forward Pass across Corpus Callosum
    optimizer.zero_grad()
    outputs = brain(lh_latents=z_lh, rh_inputs=rh_in)
    rh_updated = outputs["rh_latents_updated"]
    homeo_loss = outputs["callosum_losses"]["loss_homeostatic"]
    
    # 4. Predict Output Grid Logits: [B, 10, H, W]
    logits = arc_head(rh_updated)
    
    # 5. Masked Cross-Entropy Loss (evaluate only inside non-padding cells)
    safe_target = torch.clamp(test_target, 0, 9)
    ce_matrix = ce_loss_fn(logits, safe_target) # [B, H, W]
    task_loss = (ce_matrix * mask).sum() / (mask.sum() + 1e-8)
    
    total_loss = task_loss + 0.1 * homeo_loss
    total_loss.backward()
    nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
    optimizer.step()
    
    # 6. Evaluation Metrics
    with torch.no_grad():
        preds = logits.argmax(dim=1) # [B, H, W]
        cell_acc = (((preds == safe_target).float() * mask).sum() / (mask.sum() + 1e-8)).item() * 100.0
        
        # Spectral radius of balanced square callosal out_proj
        W_eff = brain.corpus_callosum.callosum_r_to_l.out_proj.get_effective_weight().detach().cpu()
        radius = torch.abs(torch.linalg.eigvals(W_eff)).max().item()
        
    history["task_loss"].append(task_loss.item())
    history["homeo_loss"].append(homeo_loss.item())
    history["exact_match"].append(cell_acc)
    history["spec_radius"].append(radius)
    
    if step == 1 or step % 25 == 0:
        print(f"Step {step:3d} | Task Loss: {task_loss.item():.4f} | Homeo: {homeo_loss.item():.4f} | Grid Acc: {cell_acc:.1f}% | Radius: {radius:.4f}")

print("Training Complete!")

## 6. Qualitative Evaluation & Bi-Hemispheric Synthesis

Inspect the Right Hemisphere's predicted ARC output grid and generate a linguistic explanation from Llama 3.1 conditioned on the callosal feedback.

In [ ]:
eval_batch = arc_dataset.get_batch(batch_size=1, device=device)
H, W = eval_batch.target_shapes[0]

with torch.no_grad():
    tok = lh_model.tokenize(eval_batch.text_prompts)
    z_lh_eval = lh_model.extract_residual_stream(tok["input_ids"], tok["attention_mask"]).to(torch.float32)
    rh_in_eval = arc_embedder(eval_batch.test_inputs)
    out_eval = brain(lh_latents=z_lh_eval, rh_inputs=rh_in_eval)
    
    pred_logits = arc_head(out_eval["rh_latents_updated"])
    pred_grid = pred_logits.argmax(dim=1)[0, :H, :W].cpu().numpy()
    gt_grid = eval_batch.test_targets[0, :H, :W].cpu().numpy()
    in_grid = eval_batch.test_inputs[0, :H, :W].cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(in_grid, cmap=arc_cmap, vmin=0, vmax=10)
ax[0].set_title("ARC Input Grid")
ax[0].axis("off")

ax[1].imshow(gt_grid, cmap=arc_cmap, vmin=0, vmax=10)
ax[1].set_title("Ground Truth Target")
ax[1].axis("off")

ax[2].imshow(pred_grid, cmap=arc_cmap, vmin=0, vmax=10)
ax[2].set_title("Bi-Hemispheric Predicted Grid")
ax[2].axis("off")

plt.tight_layout()
plt.show()

# Generate linguistic reflection from Llama 3.1 conditioned on callosal latents
print("\n--- Left Hemisphere (Llama 3.1) Linguistic Reflection ---")
delta_lh_feedback = out_eval["lh_latents_updated"] - z_lh_eval
explanation = lh_model.generate_with_callosal_feedback(
    prompt="Explain the spatial transformation applied to the grid:",
    delta_lh=delta_lh_feedback,
    max_new_tokens=48
)
print(f"Llama Explanation: {explanation}")
print(f"Conflict Score: {out_eval['conflict_score'].item():.4f}")
print(f"Amygdalar Affective State: {out_eval['affective_state']}")

## 7. Saving Checkpoint to Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
ckpt_dir = '/content/drive/MyDrive/brain_ai_checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)

checkpoint = {
    'brain_state_dict': brain.state_dict(),
    'arc_embedder_state_dict': arc_embedder.state_dict(),
    'arc_head_state_dict': arc_head.state_dict(),
    'history': history
}
torch.save(checkpoint, os.path.join(ckpt_dir, 'bihemispheric_llama_arc_checkpoint.pt'))
print(f"Saved Bi-Hemispheric ARC checkpoint to {ckpt_dir}/bihemispheric_llama_arc_checkpoint.pt!")